In [5]:
import re
import sys
import os
import cv2
import numpy as np

from marsneuralzoo.models.e2emvm import E2emvm
from marsneuralzoo.models.yolo_seg import SegmentationYolo

e2emvm = E2emvm(multiview=False)
seg_yo = SegmentationYolo()

Loaded SuperPoint model


In [7]:
import os
import sys
import cv2
import itertools

# from dataengine.generator.obstacle.mask import (
#     remove_overlapped_area,
# )

debug_image_dir = './cache/debug/4'

image_name_list = os.listdir(debug_image_dir)
image_name_list.sort(key=lambda x: int(os.path.splitext(x)[0]))

image_list = []
for image_name in image_name_list:
    image_path = os.path.join(debug_image_dir, image_name)
    image = cv2.imread(image_path)
    image_list.append(image)


def split_list(input_list, chunk_size):
    """input_list를 chunk_size 크기만큼 나눠 분할된 리스트들의 리스트로 반환"""
    return [
        input_list[i : i + chunk_size]
        for i in range(0, len(input_list), chunk_size)
    ]


chunk_size = 10
split_image_lists = split_list(image_list, chunk_size)

seg_results_list = []

for split_image_list in split_image_lists:
    results = seg_yo.segment_batch(split_image_list)
    seg_results_list.extend(results)

# seg_results_list = seg_yo.segment_ba                tch(image_list)


def remove_overlapped_area(segs, confs):
    """
    Remove intersection area based on confidences
    """
    indices = range(len(segs))
    combinations = list(itertools.combinations(indices, 2))
    for i, j in combinations:
        intersection = segs[i] & segs[j] > 0
        if np.sum(intersection) > 0:
            if confs[i] > confs[j]:
                segs[j][intersection] = 0
            else:
                segs[i][intersection] = 0

    mask = np.sum(segs, axis=(1, 2)) >= 100
    filtered_segs = segs[mask]
    return filtered_segs


raw_segs_list = []
segs_list = []
for seg_results in seg_results_list:
    masks = seg_results['masks']
    confs = seg_results['scores']
    raw_segs_list.append(masks)
    masks = remove_overlapped_area(masks, confs)
    segs_list.append(masks)

Loading /media/vol/shared/obstacle/models/yolosegment/best.torchscript for TorchScript inference...


In [8]:
from dataengine.generator.obstacle.debug_utils import draw_mask

out_dir = './cache/debug'
IMG_H, IMG_W = image_list[0].shape[:2]

video_writer = cv2.VideoWriter(
    f"{out_dir}/input.mp4",
    cv2.VideoWriter_fourcc(*'mp4v'),
    20,
    (IMG_W, IMG_H),
)

frame_count = 0
for image, segs in zip(image_list, raw_segs_list):
    idx = 0
    frame = image.copy()
    for seg in segs:
        draw_mask(frame, seg, idx, 0.5)
        idx += 1

    cv2.putText(
        frame,
        f"{frame_count:03d}",
        (40, 80),
        cv2.FONT_HERSHEY_SIMPLEX,
        1,
        (0, 255, 255),
        1,
        cv2.LINE_AA,
    )
    video_writer.write(frame)
    frame_count += 1

video_writer.release()

In [4]:
from dataengine.generator.obstacle.debug_utils import (
    draw_mask,
    visualize_optical_flow,
)

from dataengine.generator.obstacle.segment_tracker import (
    SegmentTracklet,
)
from itertools import count

from typing import Dict, List, Optional
from dataengine.generator.obstacle.assignment import assign_segments


def id_image_to_debug_image(image, id_image):
    ids = np.unique(id_image.flatten())

    for id in ids:
        if id == -1:
            continue
        mask = (id_image == id).astype(np.uint8)
        draw_mask(image, mask, id, 0.8)


id_video_writer = cv2.VideoWriter(
    f"{out_dir}/id_debug.mp4",
    cv2.VideoWriter_fourcc(*'mp4v'),
    20,
    (IMG_W, IMG_H),
)

flow_video_writer = cv2.VideoWriter(
    f"{out_dir}/flow.mp4",
    cv2.VideoWriter_fourcc(*'mp4v'),
    20,
    (IMG_W, IMG_H),
)


class SegmentTracker:
    """
    This class tracks the segments detected by segment model using optical flow.
    """

    def __init__(
        self,
        cam_index: int,
        id_tracklet_dict: Dict[int, SegmentTracklet],
        max_missed_track=12,
        min_iou_thresh=0.3,
    ):
        """Initialises states with tracklet_database."""
        self.cam_index = cam_index
        self.prev_timestamp = -1
        self.prev_gray: Optional[np.ndarray] = None  # H x W

        # id_tracklet_dict[id] -> tracklet
        self.id_tracklet_dict = id_tracklet_dict

        # missed_tracklets[id] -> tracklet
        self.missed_tracklets: Dict[int, SegmentTracklet] = {}

        # tracked_tracklets[id] -> tracklet
        self.tracked_tracklets: Dict[int, SegmentTracklet] = {}
        self.grid = None

        self.max_missed_track = max_missed_track
        self.min_iou_thresh = min_iou_thresh

        # self.tvl1 = cv2.optflow.createOptFlow_DualTVL1()

    def track(self, timestamp, image, segs):
        """Takes a new camera measurements and tracks currently activated
        segment tracklets.
        :param timestamp: timestamp
        :param image: image
        :param segs: segment masks from deep model
        :return: recent tracklet ids.
        """
        curr_gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)

        if self.prev_gray is None:
            self.prev_timestamp = timestamp
            self.prev_gray = curr_gray

            h, w = image.shape[:2]
            remap = np.meshgrid(
                np.arange(w, dtype=np.float32), np.arange(h, dtype=np.float32)
            )
            self.grid = np.stack(remap, axis=2)

        flow = self._calculate_flow(self.prev_gray, curr_gray)

        debug_img = image.copy()
        visualize_optical_flow(debug_img, flow)
        cv2.putText(
            debug_img,
            f"{timestamp:03d}",
            (40, 80),
            cv2.FONT_HERSHEY_SIMPLEX,
            1,
            (0, 255, 255),
            1,
            cv2.LINE_AA,
        )
        flow_video_writer.write(debug_img)

        warping_matrix = self.grid - flow

        # project active tracklets' ids into image
        shape = image.shape[:2]
        id_image = np.full(shape, -1.0, dtype=np.float32)
        # id_conf = np.full(shape, 0.0, dtype=np.float32)

        for id, seg_trl in self.tracked_tracklets.items():
            seg = seg_trl.query_segment_mask(
                self.prev_timestamp, self.cam_index
            )
            id_image[seg > 0] = id
            # id_conf[seg > 0] = 1.0

        for id, seg_trl in self.missed_tracklets.items():
            seg = seg_trl.predicted_mask
            id_image[seg > 0] = id
            # id_conf[seg > 0] = 1.0

        # print(f'before {np.sum(id_conf)}')

        id_image = cv2.remap(
            id_image,
            warping_matrix,
            None,
            interpolation=cv2.INTER_NEAREST,
            borderMode=cv2.BORDER_REFLECT,
        )

        # id_conf = cv2.remap(
        #     id_conf,
        #     warping_matrix,
        #     None,
        #     interpolation=cv2.INTER_LINEAR,
        #     borderMode=cv2.BORDER_REFLECT,
        # )

        id_image = id_image.astype(int)
        # id_conf = id_conf.astype(bool)
        # id_image[~id_conf] = -1

        id_debug = image.copy()
        id_image_to_debug_image(id_debug, id_image)
        cv2.putText(
            id_debug,
            f"{timestamp:03d}",
            (40, 80),
            cv2.FONT_HERSHEY_SIMPLEX,
            1,
            (0, 255, 255),
            1,
            cv2.LINE_AA,
        )
        id_video_writer.write(id_debug)

        prev_ids = np.unique(id_image.flatten())
        prev_ids = prev_ids[prev_ids > -1]
        h, w = curr_gray.shape[:2]

        expected_segs = []
        for id in prev_ids:
            id_seg = id_image == id
            expected_segs.append(id_seg)

        expected_segs = np.array(expected_segs, dtype=np.uint8).reshape(
            -1, h, w
        )

        matched_idx_pair, missed_idx, unmatched_seg_idx = assign_segments(
            expected_segs, segs, self.min_iou_thresh
        )

        out_tracklet_ids = []
        self.missed_tracklets = {}
        self.tracked_tracklets = {}
        for id_idx, seg_idx in matched_idx_pair:
            id = prev_ids[id_idx]

            seg = segs[seg_idx]
            seg_trl = self.id_tracklet_dict[id]

            seg_trl.add_cam_segment_mask(timestamp, self.cam_index, seg)
            out_tracklet_ids.append(id)
            self.tracked_tracklets[id] = seg_trl

        for id_idx in missed_idx:
            id = prev_ids[id_idx]
            seg_trl = self.id_tracklet_dict[id]
            seg_trl.missed_count += 1

            if np.sum(expected_segs[id_idx]) < 100:
                continue
            if seg_trl.missed_count < self.max_missed_track:
                self.missed_tracklets[id] = seg_trl
                seg_trl.update_predicted_mask(expected_segs[id_idx])

        for seg_idx in unmatched_seg_idx:
            seg = segs[seg_idx]
            seg_trl = SegmentTracklet()
            self.id_tracklet_dict[seg_trl.id] = seg_trl
            seg_trl.add_cam_segment_mask(timestamp, self.cam_index, seg)
            out_tracklet_ids.append(seg_trl.id)

            self.tracked_tracklets[seg_trl.id] = seg_trl

        self.prev_timestamp = timestamp
        self.prev_gray = curr_gray

        return out_tracklet_ids

    def _calculate_flow(self, src_gray, dst_gray):
        flow = cv2.calcOpticalFlowFarneback(
            src_gray, dst_gray, None, 0.5, 4, 30, 3, 5, 1.2, 0
        )

        # TV-L1 Optical Flow 계산
        # flow = self.tvl1.calc(src_gray, dst_gray, None)
        return flow


_cam_to_trl_ids_dict = {}
_trl_id_to_trl_dict = {}

SegmentTracklet.id_counter = count(0)

idx0 = 0
idx1 = 1

tracker0 = SegmentTracker(idx0, _trl_id_to_trl_dict)
tracker1 = SegmentTracker(idx1, _trl_id_to_trl_dict)

target_from = 0
target_end = 1e8
if target_end > len(image_list):
    target_end = len(image_list)

out_dir = './cache/debug'
video_writer = cv2.VideoWriter(
    f"{out_dir}/0.mp4",
    cv2.VideoWriter_fourcc(*'mp4v'),
    20,
    (IMG_W, IMG_H),
)

timestamp = target_from
for i in range(target_from, target_end):
    image = image_list[i]
    segs = segs_list[i]
    curr_tracklet_ids = tracker0.track(timestamp, image, segs)

    frame = image.copy()
    frame = cv2.cvtColor(frame, cv2.COLOR_RGB2BGR)

    for id in curr_tracklet_ids:
        mask = _trl_id_to_trl_dict[id].query_segment_mask(timestamp, idx0)
        draw_mask(frame, mask, id, 0.8)
    cv2.putText(
        frame,
        f"{timestamp:03d}",
        (40, 80),
        cv2.FONT_HERSHEY_SIMPLEX,
        1,
        (0, 255, 255),
        1,
        cv2.LINE_AA,
    )

    video_writer.write(frame)
    timestamp += 1
video_writer.release()

flow_video_writer.release()
id_video_writer.release()
# video_writer = cv2.VideoWriter(
#     f"{out_dir}/1.mp4",
#     cv2.VideoWriter_fourcc(*'mp4v'),
#     20,
#     (IMG_W, IMG_H),
# )

# timestamp = target_from
# for i in reversed(range(target_from, target_end)):
#     image = image_list[i]
#     segs = segs_list[i]
#     curr_tracklet_ids = tracker1.track(timestamp, image, segs)

#     frame = image.copy()
#     frame = cv2.cvtColor(frame, cv2.COLOR_RGB2BGR)

#     for id in curr_tracklet_ids:
#         mask = _trl_id_to_trl_dict[id].query_segment_mask(timestamp, idx1)
#         draw_mask(frame, mask, id, 0.6)
#     cv2.putText(
#         frame,
#         f"{timestamp:03d}",
#         (40, 80),
#         cv2.FONT_HERSHEY_SIMPLEX,
#         1,
#         (0, 255, 255),
#         1,
#         cv2.LINE_AA,
#     )

#     video_writer.write(frame)
#     timestamp += 1


# video_writer.release()

kj/filesystem-disk-unix.c++:1690: warning: PWD environment variable doesn't match current directory; pwd = /home/mars
